# Semana 02
## Perfilado, granularidad y gobierno minimo del dato

**Objetivo**: perfilar un dataset con problemas reales o simulados y convertir ese perfilado en decisiones analiticas.

**Herramientas teoricas de la semana**
- unidad de analisis
- granularidad
- completitud y cardinalidad
- trazabilidad y metadata minima


### Agenda sugerida de 4 horas
- 0:00 - 0:30: granularidad y errores semanticos frecuentes
- 0:30 - 1:20: perfilado con `pandas`
- 1:20 - 2:10: perfilado visual y deteccion de riesgos
- 2:10 - 3:20: tabla de perfilado y preguntas viables/no viables
- 3:20 - 4:00: cierre, bitacora de datos y reflexion


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-02"
OUTPUT_DIR = ensure_output_dir(WEEK)

raw_sales = introduce_quality_issues(make_base_sales(n=2200, seed=52), seed=52)
profile = profile_dataframe(raw_sales)
profile


### Actividad 1. Perfilado tabular
Calcula para cada campo:
- tipo inferido
- porcentaje de nulos
- cardinalidad
- un ejemplo de valor
- riesgo principal para visualizacion


In [ ]:
profile.sort_values(['missing_pct', 'n_unique'], ascending=[False, False]).head(12)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
profile.sort_values('missing_pct', ascending=False).head(10).plot(
    kind='barh', x='column', y='missing_pct', ax=axes[0], color='#6b7280', legend=False, title='Top campos por porcentaje de nulos'
)
profile.sort_values('n_unique', ascending=False).head(10).plot(
    kind='barh', x='column', y='n_unique', ax=axes[1], color='#111827', legend=False, title='Top campos por cardinalidad'
)
plt.tight_layout()


### Actividad 2. Granularidad y validez de preguntas
Usa el dataset para responder:
- Una fila representa una venta, un resumen o una mezcla?
- Que preguntas son validas con esta granularidad?
- Que preguntas parecen naturales pero en realidad no pueden sostenerse?


In [ ]:
question_matrix = pd.DataFrame([
    {'question': 'Ventas por categoria', 'is_viable': True, 'reason': 'la fuente contiene categoria y ventas por transaccion'},
    {'question': 'Tiempo promedio entre orden y entrega', 'is_viable': True, 'reason': 'existen order_date y ship_date'},
    {'question': 'Nivel de satisfaccion del cliente', 'is_viable': False, 'reason': 'no hay variable observada de satisfaccion'},
    {'question': 'Causalidad entre descuento y fidelidad', 'is_viable': False, 'reason': 'no existe diseño causal ni variable de fidelidad'},
])
question_matrix


In [ ]:
save_for_tableau(raw_sales, WEEK, 'raw_sales_with_issues')
save_for_tableau(profile, WEEK, 'profile_table')


### Uso teorico de herramientas
- `pandas.dtypes` conecta con el tipo semantico de las variables.
- `nunique()` y `isna().mean()` operacionalizan cardinalidad y completitud.
- Las tablas exportadas pueden acompañar el dashboard como documentación de calidad.
